# 02 — Modeling and Precision-Recall Tradeoff

Scores were produced by `scripts/run_modeling.py`. This notebook reads them and focuses on
threshold analysis and probability calibration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression as LR
from fraud.data import RES, FIG

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

d = np.load(RES / "scores.npz")
yte, yva = d["yte"], d["yva"]
names = ["logreg", "iforest", "xgb", "xgb_smote"]

tab = pd.read_csv(RES / "model_comparison.csv")
print(tab.to_string(index=False))

In [ ]:
colors = {"logreg": "#4878CF", "iforest": "#6ACC65",
          "xgb": "#D65F5F", "xgb_smote": "#B47CC7"}

fig, ax = plt.subplots(figsize=(7, 5))
for n in names:
    p, r, _ = precision_recall_curve(yte, d[f"{n}_te"])
    ax.plot(r, p, label=n, color=colors[n])
ax.axhline(yte.mean(), ls="--", c="grey", label=f"no skill ({yte.mean():.4f})")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("Precision-recall curves — held-out future test set")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
hours = (d["time_te"].max() - d["time_te"].min()) / 3600
s = d["xgb_te"]
rows = []
for t in (0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99):
    pred = s >= t
    tp = int((pred & (yte == 1)).sum())
    rows.append(dict(
        threshold=t,
        flagged_per_hour=round(pred.sum() / hours, 1),
        precision=round(tp / max(pred.sum(), 1), 3),
        recall=round(tp / int(yte.sum()), 3),
        frauds_missed=int(yte.sum()) - tp,
    ))
tab2 = pd.DataFrame(rows)
tab2.to_csv(RES / "threshold_table.csv", index=False)
print(tab2.to_string(index=False))

In [ ]:
p, r, t = precision_recall_curve(yte, d["xgb_te"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, p[:-1], label="precision", color="#4878CF")
ax.plot(t, r[:-1], label="recall", color="#D65F5F")
ax.set_xlabel("decision threshold")
ax.set_ylabel("score")
ax.set_title("XGBoost: precision and recall vs threshold (test set)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "threshold_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fp_, mp_ = calibration_curve(yte, d["xgb_te"], n_bins=10, strategy="quantile")

# fit isotonic on validation set
iso = IsotonicRegression(out_of_bounds="clip").fit(d["xgb_va"], yva)
cal_te_iso = iso.predict(d["xgb_te"])

# also try Platt scaling (logistic on raw score)
platt = LR().fit(d["xgb_va"].reshape(-1, 1), yva)
cal_te_platt = platt.predict_proba(d["xgb_te"].reshape(-1, 1))[:, 1]

print("Brier score (lower is better):")
print(f"  raw:      {brier_score_loss(yte, d['xgb_te']):.5f}")
print(f"  isotonic: {brier_score_loss(yte, cal_te_iso):.5f}")
print(f"  Platt:    {brier_score_loss(yte, cal_te_platt):.5f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
ax.plot(mp_, fp_, marker="o", label="XGBoost (raw)", color="#D65F5F")
fp2, mp2 = calibration_curve(yte, cal_te_iso, n_bins=10, strategy="quantile")
ax.plot(mp2, fp2, marker="s", label="isotonic calibrated", color="#4878CF")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("fraction of positives")
ax.set_title("Reliability curve — XGBoost")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "calibration_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# save calibrated scores
np.save(RES / "xgb_te_calibrated.npy", cal_te_iso)
np.save(RES / "xgb_va_calibrated.npy", iso.predict(d["xgb_va"]))

## Discussion

**XGBoost vs Logistic Regression**: The XGBoost model's higher PR-AUC reflects its ability to
capture non-linear interactions between the anonymized V-features, which are themselves already
linear combinations of the original variables.

**Isolation Forest**: It never sees fraud labels during training. Its anomaly scores correlate
with fraud but it also flags unusual-but-legitimate transactions. This shows up as lower precision
at any given recall level.

**SMOTE vs class weighting**: Whether SMOTE improves over class weighting is an empirical
question. On this dataset, with a time-based split, we report the actual result above.

**Calibration**: `scale_pos_weight` shifts predicted probabilities. After isotonic calibration on
the validation set, Brier scores improve and the reliability curve is closer to the diagonal.